# Inspect FS-TACRED `final_step_dev`

This notebook loads the few-shot episodes and displays every support and query sentence with the **subject** and **object** marked. Change `EPISODE_INDEX` or use the optional slider at the bottom to browse.

In [1]:
from pathlib import Path
import html
import pickle

from IPython.display import HTML, display

# This works whether Jupyter was started from the repository root or codes/.
def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "codes").is_dir():
            return candidate
    raise FileNotFoundError("Could not find a repository root containing data/ and codes/.")

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"

DATASET = "fs_tacred"
SPLIT = "final_step_dev"
# Set this to True to inspect the fixed 6,000-episode subset instead.
USE_6000_SUBSET = False

subset = "_6000" if USE_6000_SUBSET else ""
EPISODES_PATH = DATA_DIR / f"{DATASET}_{SPLIT}{subset}_episodes_1shots.pkl"
DETAILS_PATH = DATA_DIR / f"{DATASET}_{SPLIT}{subset}_episodes_shots_details.pkl"

with EPISODES_PATH.open("rb") as f:
    episodes = pickle.load(f)
with DETAILS_PATH.open("rb") as f:
    details = pickle.load(f)

shots = details["shots"]
queries = details["queries"]
umbc_shots = details.get("umbc_shots", {})

print(f"Loaded {len(episodes):,} episodes")
print(f"Support records: {len(shots):,}; query records: {len(queries):,}")
print(f"Episodes: {EPISODES_PATH}")
print(f"Details:  {DETAILS_PATH}")

Loaded 9,000 episodes
Support records: 633; query records: 7,365
Episodes: /storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/data/fs_tacred_final_step_dev_episodes_1shots.pkl
Details:  /storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/data/fs_tacred_final_step_dev_episodes_shots_details.pkl


## Formatting helpers

The text formatter uses `<subject>...</subject>` and `<object>...</object>`. The HTML formatter uses color highlighting while preserving those roles.

In [2]:
def marked_tokens(record: dict) -> list[str]:
    """Return tokens with subject/object XML-style markers inserted."""
    tokens = list(record["token"])
    spans = [
        (record["subj_start"], record["subj_end"], "subject"),
        (record["obj_start"], record["obj_end"], "object"),
    ]
    for start, end, role in spans:
        tokens[start] = f"<{role}>" + tokens[start]
        tokens[end] = tokens[end] + f"</{role}>"
    return tokens


def marked_text(record: dict) -> str:
    return " ".join(marked_tokens(record))


def entity_text(record: dict, role: str) -> str:
    start = record[f"{role}_start"]
    end = record[f"{role}_end"]
    return " ".join(record["token"][start : end + 1])


def marked_html(record: dict) -> str:
    """Render a record safely, with subject in blue and object in orange."""
    rendered = []
    for i, token in enumerate(record["token"]):
        safe_token = html.escape(str(token))
        roles = []
        if record["subj_start"] <= i <= record["subj_end"]:
            roles.append("subject")
        if record["obj_start"] <= i <= record["obj_end"]:
            roles.append("object")
        for role in roles:
            safe_token = f'<span class="entity {role}">{safe_token}</span>'
        rendered.append(safe_token)
    return " ".join(rendered)


CSS = """
<style>
.episode {font-family: system-ui, sans-serif; max-width: 1100px; line-height: 1.55}
.episode .legend {margin: 8px 0 14px; color: #555}
.episode .card {border: 1px solid #ddd; border-radius: 8px; padding: 12px 14px; margin: 9px 0}
.episode .query {border-left: 5px solid #7c3aed; background: #faf7ff}
.episode .support {border-left: 5px solid #64748b}
.episode .meta {font-size: 0.88rem; color: #555; margin-bottom: 5px}
.entity {border-radius: 4px; padding: 1px 4px; font-weight: 650}
.subject {background: #bfdbfe; color: #1e3a8a}
.object {background: #fed7aa; color: #7c2d12}
.relation-match {color: #15803d; font-weight: 650}
.relation-miss {color: #b91c1c; font-weight: 650}
</style>
"""

## Episode viewer

In [6]:
def get_support(support_id: str) -> dict:
    if support_id in shots:
        return shots[support_id]
    if support_id in umbc_shots:
        return umbc_shots[support_id]
    raise KeyError(f"Support ID not found: {support_id}")


def episode_records(episode_index: int):
    episode = episodes[episode_index]
    support_records = [
        get_support(support_id)
        for way in episode["meta_train"]
        for support_id in way
    ]
    query_records = [queries[query_id] for query_id in episode["meta_test"]]
    return support_records, query_records


def display_episode(episode_index: int = 0) -> None:
    if not 0 <= episode_index < len(episodes):
        raise IndexError(f"episode_index must be in [0, {len(episodes) - 1}]")

    support_records, query_records = episode_records(episode_index)
    support_relations = {record["relation"] for record in support_records}
    blocks = [
        CSS,
        '<div class="episode">',
        f"<h2>Episode {episode_index:,}</h2>",
        '<div class="legend"><span class="entity subject">Subject</span> '
        '<span class="entity object">Object</span></div>',
        "<h3>Support examples</h3>",
    ]

    for i, record in enumerate(support_records, start=1):
        blocks.extend([
            '<div class="card support">',
            f'<div class="meta">Support {i} · relation: <b>{html.escape(record["relation"])}</b> '
            f'· id: {html.escape(str(record["id"]))}</div>',
            f"<div>{marked_html(record)}</div>",
            "</div>",
        ])

    blocks.append("<h3>Query examples</h3>")
    for i, record in enumerate(query_records, start=1):
        is_positive = record["relation"] in support_relations
        label = "positive (relation is in support set)" if is_positive else "negative (relation is outside support set)"
        label_class = "relation-match" if is_positive else "relation-miss"
        blocks.extend([
            '<div class="card query">',
            f'<div class="meta">Query {i} · gold relation: <b>{html.escape(record["relation"])}</b> '
            f'· <span class="{label_class}">{label}</span> · id: {html.escape(str(record["id"]))}</div>',
            f"<div>{marked_html(record)}</div>",
            "</div>",
        ])

    blocks.append("</div>")
    display(HTML("".join(blocks)))


EPISODE_INDEX = 100
display_episode(EPISODE_INDEX)

## Inspect the same episode as a table

This is useful for copying tagged text or filtering records programmatically.

In [ ]:
def display_rows(rows: list[dict], columns: list[str] | None = None) -> None:
    """Display a list of dictionaries as a dependency-free HTML table."""
    if not rows:
        display(HTML("<i>No matching rows.</i>"))
        return
    columns = columns or list(rows[0])
    header = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(
            f"<td>{html.escape(str(row.get(column, '')))}</td>" for column in columns
        ) + "</tr>"
        for row in rows
    )
    display(HTML(
        "<style>.data-table{border-collapse:collapse;font-family:system-ui;font-size:13px}"
        ".data-table th,.data-table td{border:1px solid #ddd;padding:6px;vertical-align:top}"
        ".data-table th{background:#f5f5f5;position:sticky;top:0}</style>"
        f'<div style="overflow:auto;max-height:650px"><table class="data-table">'
        f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>"
    ))


def episode_table(episode_index: int = 0) -> list[dict]:
    support_records, query_records = episode_records(episode_index)
    rows = []
    for kind, records in (("support", support_records), ("query", query_records)):
        for position, record in enumerate(records, start=1):
            rows.append({
                "kind": kind,
                "position": position,
                "id": record["id"],
                "relation": record["relation"],
                "subject": entity_text(record, "subj"),
                "object": entity_text(record, "obj"),
                "subject_type": record.get("subj_type"),
                "object_type": record.get("obj_type"),
                "marked_sentence": marked_text(record),
            })
    return rows


episode_rows = episode_table(EPISODE_INDEX)
display_rows(episode_rows)

## Find episodes by query relation

Pass a TACRED relation such as `"per:age"`, or `"no_relation"`.

In [ ]:
def find_episodes(query_relation: str, limit: int = 20) -> list[dict]:
    matches = []
    for episode_index, episode in enumerate(episodes):
        for query_id in episode["meta_test"]:
            query = queries[query_id]
            if query["relation"] == query_relation:
                support_relations = [
                    get_support(support_id)["relation"]
                    for way in episode["meta_train"]
                    for support_id in way
                ]
                matches.append({
                    "episode_index": episode_index,
                    "query_id": query_id,
                    "query_relation": query_relation,
                    "positive": query_relation in support_relations,
                    "support_relations": support_relations,
                    "query": marked_text(query),
                })
                if len(matches) >= limit:
                    return matches
    return matches


matches = find_episodes("per:age", limit=10)
display_rows(matches)

## Optional interactive browser

If `ipywidgets` is installed, this creates a slider for moving through episodes.

In [ ]:
try:
    import ipywidgets as widgets

    widgets.interact(
        display_episode,
        episode_index=widgets.IntSlider(
            value=EPISODE_INDEX,
            min=0,
            max=len(episodes) - 1,
            step=1,
            description="Episode",
            continuous_update=False,
            layout=widgets.Layout(width="800px"),
        ),
    )
except ImportError:
    print("ipywidgets is not installed; call display_episode(index) directly.")